# 5장.Smolagents 에이전트: ACP 서버 래핑

## 에이전트를 ACP 서버에 래핑

In [3]:
%%writefile acp_project/guideagent_server.py

from collections.abc import AsyncGenerator
from acp_sdk.models import Message, MessagePart
from acp_sdk.server import Context, RunYield, RunYieldResume, Server
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, VisitWebpageTool
import logging
from dotenv import load_dotenv

load_dotenv()

server = Server()

model = LiteLLMModel(
    model_id="openai/gpt-4o-mini",
    max_tokens=2048
)

@server.agent()
async def guide_agent(input: list[Message], context: Context) -> AsyncGenerator[RunYield, RunYieldResume]:
    
    """응급환자 발생시 관련 질문을 검색하여 지원하는 Agent다. 교내의 응급 발생 시 교직원과 학생이 응급 처치 방법을 검색하여 찾는 데 사용할 수 있다."""
    agent = CodeAgent(tools=[DuckDuckGoSearchTool(), VisitWebpageTool()], model=model)

    prompt = input[0].parts[0].content
    response = agent.run(prompt)

    yield Message(parts=[MessagePart(content=str(response))])


if __name__ == "__main__":
    server.run(port=8001)

Writing acp_project/guideagent_server.py


## 터미널에서 Guide ACP 서버 실행

```
uv run guideagent_server.py
```